In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import logging
import urllib


for name in ("httpx", "httpx2", "httpcore", "openai"):
    logging.getLogger(name).setLevel(logging.WARNING)

In [111]:
from dataclasses import dataclass
from langchain.agents import AgentState, create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents.middleware import wrap_model_call, dynamic_prompt, HumanInTheLoopMiddleware, SummarizationMiddleware
from langchain.agents.middleware import ModelRequest, ModelResponse
from typing import Callable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_unstructured import UnstructuredLoader
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
from pprint import pprint
from langchain.messages import HumanMessage
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langchain.agents.middleware import dynamic_prompt
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
loader = UnstructuredLoader("documents/company_faq.md")

data = loader.load()

print(data)

[Document(metadata={'source': 'documents/company_faq.md', 'category_depth': 0, 'languages': ['eng'], 'file_directory': 'documents', 'filename': 'company_faq.md', 'filetype': 'text/markdown', 'last_modified': '2026-10-08T13:52:40', 'category': 'Title', 'element_id': '30571a8251483ad42feee96d9bab6a5e'}, page_content='Khalid Electronics — Company FAQ'), Document(metadata={'source': 'documents/company_faq.md', 'category_depth': 1, 'languages': ['eng'], 'file_directory': 'documents', 'filename': 'company_faq.md', 'filetype': 'text/markdown', 'last_modified': '2026-10-08T13:52:40', 'parent_id': '30571a8251483ad42feee96d9bab6a5e', 'category': 'Title', 'element_id': '59dfdb7f393c99852a9dec8b7a17f146'}, page_content='About the Business'), Document(metadata={'source': 'documents/company_faq.md', 'languages': ['eng'], 'file_directory': 'documents', 'filename': 'company_faq.md', 'filetype': 'text/markdown', 'last_modified': '2026-10-08T13:52:40', 'parent_id': '59dfdb7f393c99852a9dec8b7a17f146', 'c

In [5]:
loader = DirectoryLoader(
    "documents",
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
    show_progress=True
)

data = loader.load()
print(f"Loaded {len(data)} documents")

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00,  8.77it/s]

Loaded 7 documents


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    add_start_index=True
)

splits = text_splitter.split_documents(data)

print(f"Created {len(splits)} chunks")

Created 14 chunks


In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embeddings
)

print("Vector store initialized successfully.")

Vector store initialized successfully.


In [8]:
ids = vector_store.add_documents(documents=splits)

result = vector_store.similarity_search(
    "Customer support"
)

print(result[0])

page_content='For security, customers cannot use support to access another customer's order information.

Changing an Order

Customers should contact support as soon as possible if they need to change an order. Changes are not guaranteed after an order has been shipped.

Out-of-Scope Questions

If information cannot be verified from the business documents or authorized systems, the support agent should not invent an answer. It should explain that the information needs confirmation from a staff member.' metadata={'source': 'documents\\company_faq.md', 'start_index': 1370}


In [9]:
for doc in data:
    print(doc.metadata.get("source"))

documents\company_faq.md
documents\discount_policy.md
documents\product_catalog.md
documents\refund_policy.md
documents\return_policy.md
documents\shipping_policy.md
documents\warranty_policy.md


In [10]:
@tool
def search_faq(query: str) -> str:
    """Search business FAQ and policy documents for relevant information."""

    results = vector_store.similarity_search(
        query,
        k=3
    )

    if not results:
        return "No relevant information was found in the business documents."

    formatted_results = []

    for doc in results:
        source = doc.metadata.get("source", "Unknown source")

        formatted_results.append(
            f"Source: {source}\n"
            f"Content:\n{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted_results)

In [11]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[search_faq],
    system_prompt="You are a helpful agent that can search for information from the retrived document. if the answer isn't in the retrieved text, tell the customer and offer to pass it to a human."
)

In [12]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Can I return a damaged laptop after 14 days, and will delivery charges be refunded?")]
    }
)

pprint(response)

{'messages': [HumanMessage(content='Can I return a damaged laptop after 14 days, and will delivery charges be refunded?', additional_kwargs={}, response_metadata={}, id='33456937-7cbe-42d9-ac50-87abc9113f96'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 187, 'total_tokens': 414, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXtOvfEcECaRynQtWcTSJMycw7cg7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c69-c873-7530-98f0-9acebcc128e0-0', tool_calls=[{'name': 'search_faq', 'args

In [13]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="What is the price of a product?")]
    }
)

pprint(response)

{'messages': [HumanMessage(content='What is the price of a product?', additional_kwargs={}, response_metadata={}, id='c504f9b8-0a95-4967-9cc3-d96d07f5bed3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 315, 'prompt_tokens': 177, 'total_tokens': 492, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXtPPKLeWXlsuPtUyVVI7HFjJasKN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c6a-3b1b-78f3-b5e5-bab98910c4d1-0', tool_calls=[{'name': 'search_faq', 'args': {'query': 'price of a product'}, 'id': 'call_WJp6

In [75]:
import urllib.parse
from langchain_community.utilities import SQLDatabase

connection_string = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-PMLLD8S;"
    "DATABASE=cus_ag;"
    "Trusted_Connection=yes;"
)

db = SQLDatabase.from_uri(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

print(db.get_usable_table_names())

['ApprovalRequests', 'OrderItems', 'Orders', 'RefundRequests', 'customers', 'products']


In [14]:
import urllib
from datetime import date, datetime
from sqlalchemy import create_engine, text

connection_string = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-PMLLD8S;"
    "DATABASE=cus_ag;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")
db = SQLDatabase(engine)

print(db.get_usable_table_names())

['ApprovalRequests', 'OrderItems', 'Orders', 'RefundRequests', 'customers', 'products']


In [ ]:
@tool
def cus_db(query: str) -> str:
    """Gather information from the database"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database {e}"

In [122]:
NO_CUSTOMER = "No customer is logged in, so I can't look up account information."

@tool
def get_order_details(order_id: str, runtime: ToolRuntime[SupportContext]) -> str:
    """Get an order's Orderstatus, total and customer from its OrderID."""
    cid = ctx_value(runtime, "customer_id")
    if not cid:
        return NO_CUSTOMER
    return db.run(
        "SELECT o.OrderID, o.OrderStatus, o.TotalAmount, c.CustomerID, c.FirstName, c.LastName "
        "FROM Orders o JOIN Customers c ON c.CustomerID = o.CustomerID WHERE o.OrderID = :oid",
        parameters={"oid": order_id, "cid": cid}
        ) or "No order with that ID was found for this customer."

@tool
def list_customer_orders(runtime: ToolRuntime[SupportContext]) -> str:
    """List all orders (ID, Orderstatus, total) for a CustomerID."""
    cid = ctx_value(runtime, "customer_id")
    if not cid:
        return NO_CUSTOMER
    return db.run("SELECT TOP 20 OrderID, OrderStatus, TotalAmount FROM Orders "
                "WHERE CustomerID = :cid ORDER BY OrderDate DESC",
                parameters={"cid": cid}) or "No orders found."

@tool
def get_customer_spending(runtime: ToolRuntime[SupportContext]) -> str:
    """Number of orders and total amount spent by the current customer."""
    cid = ctx_value(runtime, "customer_id")
    if not cid:
        return NO_CUSTOMER
    return db.run(
        "SELECT COUNT(*) AS Orders, COALESCE(SUM(TotalAmount), 0) AS TotalSpent "
        "FROM Orders WHERE CustomerID = :cid",
        parameters={"cid": cid},
    )

@tool
def check_product_stock(product_name: str) -> str:
    """Check the price and stock quantity of products whose name matches the given text."""
    return db.run(
        "SELECT TOP 5 ProductName, Brand, Category, Price, StockQuantity "
        "FROM Products WHERE ProductName LIKE :pn",
        parameters={"pn": f"%{product_name}%"},
    ) or "No matching product found."

In [ ]:
@dataclass
class SupportContext:
    customer_id:str
    tier:str ="standard"

@tool
def get_order_status(order_id: str, runtime: ToolRuntime[SupportContext]) -> str:
    """Look up the status of one of the current customer's orders."""
    result = db.run(
        "SELECT OrderID, OrderStatus, TotalAmount FROM Orders "
        "WHERE OrderID = :oid AND CustomerID = :cid",
        parameters={"oid": order_id, "cid": runtime.context.customer_id},
    )
    return result or "No order with that ID found for this customer."

@tool
def list_my_order(runtime: ToolRuntime[SupportContext]) -> str:
    """List all orders (ID, OrderStatus, Total) by a customer"""
    return db.run(
        "SELECT OrderID, OrderStatus, TotalAmount FROM Orders WHERE CustomerID= :cid",
        parameters={"cid": runtime.context.customer_id}
    ) or "No orders found."

@tool
def get_my_spending(runtime: ToolRuntime[SupportContext]) -> str:
    """Total numbers of order and Total amount spent by the customer."""
    return db.run(
        "SELECT COUNT(*), SUM(TotalAmount) FROM Orders WHERE CustomerID= :cid",
        parameters={"cid": runtime.context.customer_id}
    )
    

In [24]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[list_customer_orders, get_order_details, get_customer_spending, check_product_stock],
    system_prompt="""
    You are a helpful customer assistance for an online store.
    - use list_customer_orders for "my orders" requests.
    - use get_order_details when given an OrderID.
    - use get_customer_spending fot total and amount spent.
    - use check_product_stock to check if the product it's available also return the price, category and
    Always call a tool for order data, never guess. if a tool returns nothings, say so.
    Be brief and friendly.
    """
)

In [25]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Waht is the status of my order ORD00009.")]
    }
)

pprint(response)

{'messages': [HumanMessage(content='Waht is the status of my order ORD00009.', additional_kwargs={}, response_metadata={}, id='0e690bba-9288-464a-9507-8933adcdf2db'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 338, 'total_tokens': 493, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXtW0Y785j8WPz0Aqqqkouep9HH8R', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c70-7d7c-7d53-ba75-5a49d52a6d87-0', tool_calls=[{'name': 'get_order_details', 'args': {'order_id': 'ORD00009'}, 'id': '

In [26]:
print(response["messages"][-1].content)

ORD00009 is currently Processing.  
Order total: 220,019.00  
Customer: C00009 (Yusuf Abdullahi)

Need me to check for updates or get more details?


In [27]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="I want a refund for my order ORD00145?")]
    }
)

pprint(response)

{'messages': [HumanMessage(content='I want a refund for my order ORD00145?', additional_kwargs={}, response_metadata={}, id='845c64d5-b463-43a0-a189-57a146665169'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 337, 'total_tokens': 620, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXtWKmw9XWg6sfloPtHxKxSnXxzTb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c70-cb2c-7591-951c-584962eef209-0', tool_calls=[{'name': 'get_order_details', 'args': {'order_id': 'ORD00145'}, 'id': 'ca

In [28]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="I want to know about the availabity of this product Tablet HP Series 3?")]
    }
)

pprint(response)

{'messages': [HumanMessage(content='I want to know about the availabity of this product Tablet HP Series 3?', additional_kwargs={}, response_metadata={}, id='bec7e7d9-49bb-4640-9256-9d0eb8b2dade'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 413, 'prompt_tokens': 344, 'total_tokens': 757, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXtXSeRgJuwCWge0h5WNXXmwUq5U7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c71-db4a-7203-b9b7-b7304b164fd8-0', tool_calls=[{'name': 'check_product_stock', 'args': 

In [29]:
@dataclass
class SupportContext:
    customer_id:str
    tier : str="standard"

In [60]:
import datetime as dt

In [65]:

refund_window_days = 14

def _to_date_(value):
    if isinstance(value, dt.datetime):
        return value.date()
    if isinstance(value, str):
        return dt.date.fromisoformat(value[:10])
    return value

def _check_eligibility_(order_id:str, customer_id:str) -> tuple[bool, str]:
    with engine.connect() as conn:
        order = conn.execute(
            text("SELECT OrderStatus, OrderDate FROM Orders "
                 "WHERE OrderID= :oid AND CustomerID = :cid"),
                 {"oid": order_id, "cid": customer_id}
        ).fetchone()

        if order is None:
            return False, "No order with that ID is found for this customer"

        status, order_date = order
        if status.lower() != "delivered":
            return False, f"This order is '{status}', so it can't be refunded yet."

        existing = conn.execute(
            text("SELECT 1 FROM RefundRequests "
                 "WHERE OrderID = :oid AND Status IN ('Pending', 'Approved')"),
                 {"oid": order_id, "cid": customer_id}
        ).fetchone()

        if existing:
            return False, "A request refund already exists for this order."

        days = (dt.date.today() - _to_date_(order_date)).days

        if days > refund_window_days:
            return False, f"Ordered {days} days ago; the refund window is {refund_window_days} days."
        return True, f"Ordered {days} days ago, within the {refund_window_days}-day window."

In [95]:
@tool
def check_refund_eligibility(order_id:str, runtime: ToolRuntime[SupportContext]):
    """Check whether the current customer's order is eligible for a refund."""
    eligible, reason = _check_eligibility_(order_id, runtime.context.customer_id)
    return f"{'ELIGIBLE' if eligible else 'NOT ELIGIBLE'}: {reason}"

@tool
def request_refund(order_id:str, reason: str, runtime: ToolRuntime[SupportContext]):
    """Record a refund request for the current customer's order."""
    customer_id = runtime.context.customer_id

    eligibe, why = _check_eligibility_(order_id, customer_id)
    if not eligibe:
        return f"Refund request not recorded: {why}"

    with engine.begin() as conn:
        conn.execute(
            text(
                "INSERT INTO RefundRequests (OrderID, CustomerID, Reason, Status, RequestedAt) "
                "VALUES (:oid, :cid, :reason, 'pending', :ts)"),
            {"oid": order_id, "cid": customer_id, "reason": reason, "ts": dt.datetime.now()},
            
        )

    return f"Refund request is recorded for {order_id} and is under review."
        

In [97]:
refund_prompt = """You are a helpful assistance that can check user refund eligibilty.
yor first step is to check the refund policy to approve user refund"""

@dynamic_prompt
def refund_prompt_fn(request: ModelRequest) -> str:
    return refund_prompt

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_faq, get_order_details, check_refund_eligibility, request_refund],
    context_schema=SupportContext,
    checkpointer=InMemorySaver(),    
    middleware=[
        refund_prompt_fn,              
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_faq": False,
                "get_order_details": False,
                "check_refund_eligibility": False,
                "request_refund": True,
            }
        ),
    ],
)

In [68]:
config = {"configurable": {"thread_id": "t1"}}
ctx = SupportContext(customer_id="C00003")

response = agent.invoke({"messages": [HumanMessage(content="I want a refund for ORD00003")]},
             config=config, context=ctx)

pprint(response)

{'messages': [HumanMessage(content='I want a refund for ORD00003', additional_kwargs={}, response_metadata={}, id='51e0aef2-6cd5-43b1-ba94-26ff8eb3a951'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 384, 'prompt_tokens': 260, 'total_tokens': 644, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXuFESNDIheAJuIDIpsZ6dh2iOMvC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c9b-4301-7613-bdeb-848d39a815b7-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {'order_id': 'ORD00003'}, 'id': 'call_

In [69]:
config = {"configurable": {"thread_id": "1"}}
ctx = SupportContext(customer_id="C00007")

response = agent.invoke({"messages": [HumanMessage(content="I want a refund for ORD00007")]},
             config=config, context=ctx)

pprint(response)

{'messages': [HumanMessage(content='I want a refund for ORD00007', additional_kwargs={}, response_metadata={}, id='e2225a13-c85a-410c-8b8a-7d7afa58f4f2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 510, 'prompt_tokens': 260, 'total_tokens': 770, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXuG3lD6gJx8g2tDbkw7Jc2BQDUTC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12c9c-0b75-76f2-94b5-9ce9752bf844-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {'order_id': 'ORD00007'}, 'id': 'call_

In [82]:
print(response.get("__interrupt__"))

[Interrupt(value={'action_requests': [{'name': 'request_refund', 'args': {'order_id': 'ORD00007', 'reason': 'Damaged on arrival'}, 'description': "Tool execution requires approval\n\nTool: request_refund\nArgs: {'order_id': 'ORD00007', 'reason': 'Damaged on arrival'}"}], 'review_configs': [{'action_name': 'request_refund', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='8f4fb70968ebb046cf041df3db628444', response_schema=None)]


In [108]:
config = {"configurable": {"thread_id": "3"}}
ctx = SupportContext(customer_id="C00007")

response = agent.invoke(
    {"messages": [HumanMessage(content="I want a refund for ORD00007, the product arrived damaged")]},
    config=config, context=ctx,
)
print(response.get("__interrupt__"))

[Interrupt(value={'action_requests': [{'name': 'request_refund', 'args': {'order_id': 'ORD00007', 'reason': 'Product arrived damaged'}, 'description': "Tool execution requires approval\n\nTool: request_refund\nArgs: {'order_id': 'ORD00007', 'reason': 'Product arrived damaged'}"}], 'review_configs': [{'action_name': 'request_refund', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='c57f3a25881f59f2ece8f0859d857677', response_schema=None)]


In [109]:
response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    context=ctx
)
print(response["messages"][-1].content)

You’re all set. Here’s what happened:

- Order ORD00007: status Delivered, value 172,237.00, customer C00007, Oluwaseun Ojo
- Eligibility: Within 14-day window; order delivered 6 days ago
- Reason for refund: Product arrived damaged
- Action taken: Refund request has been recorded and is now under review by the business owner

What to expect next:
- The business owner will review the request. If approved, the refund will be processed through the original payment method.
- If any additional information is needed, you’ll be contacted.

If you’d like, I can add any extra notes to the request (e.g., attach photos of the damage, confirm packaging condition, etc.).


In [103]:
print(_check_eligibility_("ORD00001", "C00001"))

(True, 'Ordered 13 days ago, within the 14-day window.')


In [105]:
config = {"configurable": {"thread_id": "t5"}}
ctx = SupportContext(customer_id="C00001")

response = agent.invoke(
    {"messages": [HumanMessage(content="I want a refund for ORD00001, the product arrived damaged")]},
    config=config, context=ctx,
)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'order_id': 'ORD00001',
                                                                  'reason': 'Product '
                                                                            'arrived '
                                                                            'damaged'},
                                                         'description': 'Tool '
                                                                        'execution '
                                                                        'requires '
                                                                        'approval\n'
                                                                        '\n'
                                                                        'Tool: '
                                                                        'request_refund\n'
                                                                 

In [106]:
response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    context=ctx
)
print(response["messages"][-1].content)

Good news: Your refund request for ORD00001 (Damaged product) has been submitted and is currently under review.

What happens next:
- Our team will review the damage claim and the order details.
- You’ll be notified as soon as the decision is made (approval or request for more information).
- If approved, the refund will be issued to your original payment method. Processing times vary by bank/provider.

Tips to help speed things up:
- If you have photos or a short description of the damage, you can share them to support the claim.

Would you like me to also submit a refund request for ORD00007 (damaged product)? It was eligible within the 14-day window as well. If you want me to proceed, I can initiate that now.


In [110]:
print(db.run("SELECT * FROM RefundRequests"))

[(1, 'ORD00001', 'C00001', 'Product arrived damaged', 'pending', datetime.datetime(2026, 10, 11, 22, 48, 43, 203000)), (2, 'ORD00007', 'C00007', 'Product arrived damaged', 'pending', datetime.datetime(2026, 10, 11, 22, 51, 19, 983000))]


In [23]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Check refund policy eligibility"""

    request = request.override(tools=[*request.tools, search_faq])
    return handler(request)

In [24]:
refund_prompt = """You are a helpful assistance that can check user refund eligibilty.
yor first step is to check the refund policy to approve user refund"""

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:

    request = request.override(tools=[*request.tools, search_faq])
    return refund_prompt

In [35]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[check_refund_eligibility, approve_refund, search_faq],
    checkpointer=InMemorySaver(),
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "check_refund_eligibility": False,
                "search_faq": False,
                "approve_refund": True,
            })
    ]
)

In [26]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="I want a refund for my order ORD00145?")]
    },
    config={
        "configurable": {"thread_id":"1"}
    }
)

pprint(response)

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content='I want a refund for my order ORD00145?', additional_kwargs={}, response_metadata={}, id='39f5c074-90e3-4fdf-bd0a-1db77745e46c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 215, 'prompt_tokens': 220, 'total_tokens': 435, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXZOxNtepTjnOOHSXeOcnRjdd8jfL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a127d4-b87c-7490-8e12-dcffb0952ba5-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {}, 'id': 'call_bKsqOyVGjBSv

In [27]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="The product arrived damaged")]
    },
    config={
        "configurable": {"thread_id":"1"}
    }
)

pprint(response)

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content='I want a refund for my order ORD00145?', additional_kwargs={}, response_metadata={}, id='39f5c074-90e3-4fdf-bd0a-1db77745e46c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 215, 'prompt_tokens': 220, 'total_tokens': 435, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXZOxNtepTjnOOHSXeOcnRjdd8jfL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a127d4-b87c-7490-8e12-dcffb0952ba5-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {}, 'id': 'call_bKsqOyVGjBSv

In [31]:
response = agent.invoke(
    Command(
        resume= {"decisions": [{"type": "approve_refund"}]}
    ),
    config={"configurable": {"thread_id":"1"}}
)

print(response["messages"][-1].content)

Thanks for the update. Since the product arrived damaged, you likely qualify for a refund under our policy, but the refund still requires business-owner approval.

To move this forward, please provide:
- Delivery date for ORD00145
- Photos showing the damage (including the item and the packaging)
- Whether the item is unused and in its original packaging
- Any short description of the damage or defect
- Your preferred refund method (usually to the original payment method)

Once I have these, I’ll open the official refund-approval request for the business owner. After the owner approves, the refund will be processed through the original payment channel. If you’d prefer, you can also opt for a replacement instead of a refund, depending on what the owner approves.

Would you like to proceed with sending the damage evidence now?


In [133]:
def ctx_value(runtime, key, default=None):
    ctx = runtime.context
    if isinstance(ctx, dict):
        return ctx.get(key, default)
    return getattr(ctx, key, default)

In [135]:
BASE_PROMPT = """You are a customer support assistant for Khalid Electronics, an online electronics store.
- Policy and general questions: use search_faq. If the answer isn't in the retrieved text, say so and offer to pass the customer to a human.
- Order, spending and stock questions: use the database tools and report their actual results. Never guess or calculate from memory.
- Refunds: first call check_refund_eligibility. Only if it says ELIGIBLE and you have the customer's reason, call request_refund. If NOT ELIGIBLE, explain why and offer a human. Never promise a refund or offer exceptions; every request is reviewed by a person.
- Ask only for an order ID and a reason. Never ask for email, phone or card details.
- You can only access the logged-in customer's account. If asked about another customer or someone else's order, decline.
- Keep replies to 4 sentences or fewer unless the customer asks for more detail."""

TIER_NOTES = {"standard": "", "vip": "This is a VIP customer; be especially prompt and courteous."}

@dynamic_prompt
def support_prompt(request: ModelRequest) -> str:
    tier = ctx_value(request.runtime, "tier", "standard")
    return BASE_PROMPT + "\n\n" + TIER_NOTES.get(tier, "")

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_faq, get_order_details, check_refund_eligibility, request_refund, list_customer_orders, get_customer_spending, check_product_stock],
    context_schema=SupportContext,
    checkpointer=InMemorySaver(),   
    middleware=[
        support_prompt,              
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_faq": False,
                "get_order_details": False,
                "check_refund_eligibility": False,
                "request_refund": True,
            }
        ),
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 500),
            keep=("messages", 10)
        )
    ],
)

In [137]:
config = {"configurable": {"thread_id": "t7"}}
ctx = SupportContext(customer_id="C00039")

response = agent.invoke(
    {"messages": [HumanMessage(content="Yes")]},
    config=config, context=ctx,
)
pprint(response)

{'messages': [HumanMessage(content='I want a refund for ORD00039, the product arrived damaged', additional_kwargs={}, response_metadata={}, id='d7ff1824-92f1-4cb7-9617-c72a440c614b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 510, 'total_tokens': 860, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXxCU4dU6cMdSTqmZflFTWyIh7Pgc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12d48-9c81-7a02-bf5f-09e7bf2bee3a-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {'order_i

In [139]:
print(type(ctx_value))

<class 'function'>


In [141]:
config = {"configurable": {"thread_id": "t7"}}
#ctx = SupportContext(customer_id="C00039")

response = agent.invoke(
    {"messages": [HumanMessage(content="Tell me more about your company")]},
    config=config,
)
pprint(response)

{'messages': [HumanMessage(content='I want a refund for ORD00039, the product arrived damaged', additional_kwargs={}, response_metadata={}, id='d7ff1824-92f1-4cb7-9617-c72a440c614b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 510, 'total_tokens': 860, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXxCU4dU6cMdSTqmZflFTWyIh7Pgc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12d48-9c81-7a02-bf5f-09e7bf2bee3a-0', tool_calls=[{'name': 'check_refund_eligibility', 'args': {'order_i